# Logistics Transfer Delay Classification Notebook

This notebook starts the implementation of an end-to-end binary classification pipeline to predict whether a transfer is delayed by more than 30 minutes.

**Primary metric:** ROC-AUC

## 1) Data Loading

We load data with relative paths and the required file names (`train.csv`, `test.csv`).

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style='whitegrid')

data_path = 'data/'
train_df = pd.read_csv(data_path + 'train.csv')
test_df = pd.read_csv(data_path + 'test.csv')

print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')

## 2) Exploratory Data Analysis (EDA)

We inspect target balance and missingness to guide preprocessing decisions.

In [ ]:
display(train_df.head())
display(train_df.isna().mean().sort_values(ascending=False).head(10))
print(train_df['is_delayed'].value_counts(normalize=True).rename('class_ratio'))

### Visualization 1: Target distribution

This chart checks whether the delayed class is imbalanced, which affects modeling and threshold interpretation.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=train_df, x='is_delayed')
plt.title('Target Distribution: On-Time (0) vs Delayed (1)')
plt.xlabel('is_delayed')
plt.ylabel('Count')
plt.show()

**Takeaways / Insights**
- If one class is much smaller, ROC-AUC remains a robust metric for ranking quality.
- Class balance here informs future thresholding decisions, but does not change the ROC-AUC objective.

### Visualization 2: Dispatch delay by target

This chart tests whether operational delay signals separate delayed vs on-time outcomes.

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=train_df, x='is_delayed', y='dispatch_delay_minutes')
plt.title('Dispatch Delay Minutes by Target Class')
plt.xlabel('is_delayed')
plt.ylabel('dispatch_delay_minutes')
plt.show()

**Takeaways / Insights**
- A clear upward shift for the delayed class indicates strong predictive signal in dispatch delays.
- Heavy tails suggest regularized linear models may generalize better than overly complex trees.

## 3) Preprocessing

### Assumption: leakage-aware feature filtering
We remove fields that likely capture post-event outcomes too close to the target definition (actual arrival/travel metrics).

### Assumption: missing value treatment
- Numeric features use **median imputation** for robustness to outliers.
- Categorical features use **most-frequent imputation** to preserve valid category tokens for one-hot encoding.

In [ ]:
target_col = 'is_delayed'

leakage_columns = [
    'actual_arrival_time',
    'actual_travel_time_minutes',
    'in_transit_time_minutes',
]

train_work = train_df.drop(columns=leakage_columns, errors='ignore').copy()
test_work = test_df.drop(columns=leakage_columns, errors='ignore').copy()

X = train_work.drop(columns=[target_col])
y = train_work[target_col]

X_train_raw, X_valid_raw, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 4) Feature Engineering

### Assumption: time decomposition helps capture operational periodicity
We transform HHMM-style timestamps into hour/minute parts and extract calendar signals from operation date.

In [ ]:
def decompose_hhmm(df, col):
    numeric_time = pd.to_numeric(df[col], errors='coerce')
    df[f'{col}_hour'] = (numeric_time // 100).astype('float')
    df[f'{col}_minute'] = (numeric_time % 100).astype('float')

def build_features(frame):
    # Build features on a copy to avoid accidental in-place changes across train/valid/test objects.
    engineered = frame.copy()

    if 'operation_date' in engineered.columns:
        op_date = pd.to_datetime(engineered['operation_date'], errors='coerce')
        engineered['operation_month'] = op_date.dt.month
        engineered['operation_day'] = op_date.dt.day

    for time_col in ['scheduled_dispatch_time', 'scheduled_arrival_time', 'actual_dispatch_time']:
        if time_col in engineered.columns:
            decompose_hhmm(engineered, time_col)

    return engineered.drop(columns=['operation_date'], errors='ignore')

X_train = build_features(X_train_raw)
X_valid = build_features(X_valid_raw)
X_test_features = build_features(test_work)

numeric_cols = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=['number']).columns.tolist()

## 5) Modeling

We begin with an interpretable baseline (Logistic Regression), then keep a failed attempt section as required.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols),
    ]
)

# with_mean=False is required because one-hot encoding produces a sparse matrix, and centering would densify it.
log_numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler(with_mean=False)),
])

log_preprocessor = ColumnTransformer(
    transformers=[
        ('num', log_numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols),
    ]
)

logistic_pipeline = Pipeline(steps=[
    ('preprocess', log_preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

logistic_pipeline.fit(X_train, y_train)
valid_pred_log = logistic_pipeline.predict_proba(X_valid)[:, 1]
log_auc = roc_auc_score(y_valid, valid_pred_log)
print(f'Logistic Regression validation ROC-AUC: {log_auc:.4f}')

### Failed Experiment / Unsuccessful Attempt

A shallow Decision Tree was tested for interpretability but often underperforms on high-cardinality one-hot encoded tabular features.

In [ ]:
tree_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', DecisionTreeClassifier(max_depth=4, random_state=42))
])

tree_pipeline.fit(X_train, y_train)
valid_pred_tree = tree_pipeline.predict_proba(X_valid)[:, 1]
tree_auc = roc_auc_score(y_valid, valid_pred_tree)
print(f'Decision Tree validation ROC-AUC: {tree_auc:.4f}')

## 6) Evaluation

We compare validation ROC-AUC and keep the best model for full-train fitting and test prediction.

In [ ]:
model_scores = {
    'logistic_regression': log_auc,
    'decision_tree_failed_attempt': tree_auc,
}

best_name = max(model_scores, key=model_scores.get)
print('Validation ROC-AUC results:', model_scores)
print('Selected model:', best_name)

selected_pipeline = logistic_pipeline if best_name == 'logistic_regression' else tree_pipeline

## 7) Final Training and Submission File

We retrain the selected pipeline on all training rows and create predictions in the exact required format (`transfer_id`, `predict_prob`).

In [ ]:
X_full_raw = train_work.drop(columns=[target_col]).copy()
y_full = train_work[target_col].copy()

X_full = build_features(X_full_raw)

selected_pipeline.fit(X_full, y_full)
test_pred = selected_pipeline.predict_proba(X_test_features)[:, 1]

submission = pd.DataFrame({
    'transfer_id': test_df['transfer_id'],
    'predict_prob': test_pred,
})

output_path = 'output/'
from pathlib import Path
Path(output_path).mkdir(parents=True, exist_ok=True)
submission.to_csv(output_path + 'submission.csv', index=False)
display(submission.head())
print('Saved output/submission.csv with required columns.')